# Stacking 3-base — ZIT bag + LGBM single + ElasticNet single

**Base 모델 (Level 0) — 3종**:
- **ZIT bag**: BagZITboost (combined_best 신규 우선, 미실행 시 기존 best 자동 fallback)
- **LGBM single**: `4_output/final/reg_only/lgbm`
- **ElasticNet single**: `4_output/final/reg_only/enet`  (학습 시 이미 scaling 적용됨)

**Meta-learner (Level 1)**:
- **ElasticNet** (`alpha`, `l1_ratio` 작은 grid sweep)
- 입력에 **StandardScaler** 적용 (3개 base 예측 정규화)

**비교군**:
- 단일 best (3개 중 OOF 최저)
- Blending SLSQP (3개 가중평균)

**격리**: 신규 산출물 `_temp/stacking_3base/`. 기존 모듈/노트북 무수정. `final.modules.blending` import 만 사용.

## 1. 환경 + import

In [1]:
import os, sys, json

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import blending   # SLSQP baseline

from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. ZIT bag 자동 선택 + 3 base 산출물 로드

ZIT bag 우선순위: `combined_best` (신규) → `pp_hpo` → `hpo` → `fixed_ge`

처음으로 산출물 모두 갖춘 디렉토리 1개 채택. `OVERRIDE_ZIT_BAG`을 명시하면 그 이름으로 강제.

In [2]:
# 사용자 override (None 이면 자동 선택)
OVERRIDE_ZIT_BAG = None   # 예: 'bag_zit_combined_best' 강제하려면 이름 입력

ZIT_BAG_PRIORITY = [
    'bag_zit_combined_best',     # 신규 (사용자가 combined_best 노트북 실행 후)
    'bag_zit_combined_best_xy',  # 신규 xy 추가 버전
    'bag_zit_pp_hpo',             # 기존 PP HPO
    'bag_zit_hpo',                # 기존 HP HPO
    'bag_zit_fixed_ge',           # 기존 GTE 실험
]

REQUIRED = ['oof_unit.csv', 'val_unit.csv', 'test_unit.csv']

def _has_all(base):
    return all(os.path.exists(os.path.join(base, f)) for f in REQUIRED)

# ZIT bag 1개 자동 선택
zit_bag_name = None
zit_bag_path = None
candidates = [OVERRIDE_ZIT_BAG] if OVERRIDE_ZIT_BAG else ZIT_BAG_PRIORITY
for cand in candidates:
    if cand is None:
        continue
    p = os.path.join(OUTPUT_DIR, '_temp', cand)
    if _has_all(p):
        zit_bag_name = cand
        zit_bag_path = p
        break

if zit_bag_name is None:
    raise RuntimeError(f'ZIT bag 산출물 없음. 후보 디렉토리: {ZIT_BAG_PRIORITY}')

# 3 base 경로 dict
BASE_PATHS = {
    'zit_bag':  zit_bag_path,
    'lgbm':     os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm'),
    'enet':     os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'enet'),
}

print(f'=== Base 모델 3종 ===')
print(f'  zit_bag = {zit_bag_name}  →  {zit_bag_path}')
for n, p in list(BASE_PATHS.items())[1:]:
    print(f'  {n:8s}  →  {p}')

# 산출물 존재 확인
for n, p in BASE_PATHS.items():
    if not _has_all(p):
        miss = [f for f in REQUIRED if not os.path.exists(os.path.join(p, f))]
        raise RuntimeError(f'{n} ({p}) missing: {miss}')
print('\n  모든 base 산출물 OK')

=== Base 모델 3종 ===
  zit_bag = bag_zit_combined_best  →  c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_combined_best
  lgbm      →  c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\lgbm
  enet      →  c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\enet

  모든 base 산출물 OK


## 3. OOF/val/test 로드 + ufs_serial 정합 검증

In [3]:
_, ys = load_all()
y_train_unit = ys['train'][[KEY_COL, TARGET_COL]].copy()
y_val_unit   = ys['validation'][[KEY_COL, TARGET_COL]].copy()
y_test_unit  = ys['test'][[KEY_COL, TARGET_COL]].copy()

oof_dfs, val_dfs, test_dfs = {}, {}, {}
for n, base in BASE_PATHS.items():
    oof_dfs[n]  = pd.read_csv(os.path.join(base, 'oof_unit.csv'))[[KEY_COL, 'pred']]
    val_dfs[n]  = pd.read_csv(os.path.join(base, 'val_unit.csv'))[[KEY_COL, 'pred']]
    test_dfs[n] = pd.read_csv(os.path.join(base, 'test_unit.csv'))[[KEY_COL, 'pred']]

def _check_keys(dfs, expected_keys, split_name):
    expected = set(expected_keys.tolist())
    for name, df in dfs.items():
        keys = set(df[KEY_COL].tolist())
        miss = expected - keys
        extra = keys - expected
        if miss or extra:
            raise ValueError(f'{split_name}/{name}: missing={len(miss)}, extra={len(extra)}')

_check_keys(oof_dfs,  y_train_unit[KEY_COL].values, 'oof')
_check_keys(val_dfs,  y_val_unit[KEY_COL].values,   'val')
_check_keys(test_dfs, y_test_unit[KEY_COL].values,  'test')

y_oof_arr  = y_train_unit.set_index(KEY_COL)[TARGET_COL]
y_val_arr  = y_val_unit.set_index(KEY_COL)[TARGET_COL]
y_test_arr = y_test_unit.set_index(KEY_COL)[TARGET_COL]

# 정렬된 prediction 행렬 (각 split, 행=unit, 열=base 모델)
P_oof  = pd.DataFrame({n: oof_dfs[n].set_index(KEY_COL)['pred'].reindex(y_oof_arr.index)
                       for n in BASE_PATHS})
P_val  = pd.DataFrame({n: val_dfs[n].set_index(KEY_COL)['pred'].reindex(y_val_arr.index)
                       for n in BASE_PATHS})
P_test = pd.DataFrame({n: test_dfs[n].set_index(KEY_COL)['pred'].reindex(y_test_arr.index)
                       for n in BASE_PATHS})

print(f'P_oof:  {P_oof.shape},  P_val: {P_val.shape},  P_test: {P_test.shape}')
print(f'\n[정합 OK] 3 base 모델 모두 동일 ufs_serial set')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
P_oof:  (26187, 3),  P_val: (8727, 3),  P_test: (8729, 3)

[정합 OK] 3 base 모델 모두 동일 ufs_serial set


## 4. 단일 base RMSE + residual correlation

In [4]:
def _rmse(p, y):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

rows = []
for n in BASE_PATHS:
    rows.append({
        'model': n,
        'oof':   _rmse(P_oof[n].values,  y_oof_arr.values),
        'val':   _rmse(P_val[n].values,  y_val_arr.values),
        'test':  _rmse(P_test[n].values, y_test_arr.values),
    })
single_df = pd.DataFrame(rows).sort_values('oof')
print('=== 단일 base RMSE (OOF 오름차순) ===')
print(single_df.to_string(index=False, float_format='%.6f'))

# residual correlation
R_oof = P_oof.subtract(y_oof_arr, axis=0)
corr_oof = R_oof.corr()
print('\n=== residual correlation (OOF) ===')
print(corr_oof.round(4).to_string())

=== 단일 base RMSE (OOF 오름차순) ===
  model      oof      val     test
zit_bag 0.008251 0.005710 0.008412
   lgbm 0.008257 0.005731 0.008429
   enet 0.008286 0.005781 0.008459

=== residual correlation (OOF) ===
         zit_bag    lgbm    enet
zit_bag   1.0000  0.9985  0.9951
lgbm      0.9985  1.0000  0.9968
enet      0.9951  0.9968  1.0000


## 5. Blending baseline (SLSQP, 3-base 가중평균)

stacking 효과 비교용. SLSQP 가중평균 w≥0, Σw=1.

In [5]:
fit_blend = blending.fit_and_apply(
    train_preds=oof_dfs,
    val_preds=val_dfs,
    test_preds=test_dfs,
    y_train_unit=y_train_unit,
    method='slsqp',
)
w_blend = fit_blend['weights']
blend_oof  = fit_blend['train_blend']
blend_val  = fit_blend['val_blend']
blend_test = fit_blend['test_blend']

rmse_blend_oof  = _rmse(blend_oof.set_index(KEY_COL).loc[y_oof_arr.index, 'pred'].values,  y_oof_arr.values)
rmse_blend_val  = _rmse(blend_val.set_index(KEY_COL).loc[y_val_arr.index, 'pred'].values,  y_val_arr.values)
rmse_blend_test = _rmse(blend_test.set_index(KEY_COL).loc[y_test_arr.index, 'pred'].values, y_test_arr.values)

print('=== Blending (SLSQP) ===')
for n, w in sorted(w_blend.items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 50)
    print(f'  {n:10s}: {w:.4f}  {bar}')
print(f'\n  OOF :  {rmse_blend_oof:.6f}')
print(f'  val :  {rmse_blend_val:.6f}')
print(f'  test:  {rmse_blend_test:.6f}')

[blend SLSQP] weights={'zit_bag': 0.3333, 'lgbm': 0.3333, 'enet': 0.3333}, rmse=0.008256, converged=True
=== Blending (SLSQP) ===
  zit_bag   : 0.3333  ████████████████
  lgbm      : 0.3333  ████████████████
  enet      : 0.3333  ████████████████

  OOF :  0.008256
  val :  0.005730
  test:  0.008425


## 6. Stacking — ElasticNet meta-learner (with StandardScaler)

**구조**:
1. 입력: 3 base OOF 예측 (3개 컬럼)
2. StandardScaler 로 정규화 (mean=0, std=1)
3. ElasticNet 으로 학습 (alpha, l1_ratio 그리드 탐색)
4. val/test 의 base 예측에도 동일 transform → predict
5. 음수 예측은 0으로 clip (health ≥ 0)

**Overfit 방지**: ElasticNet 의 alpha 가 OOF 노이즈를 흡수하지만, 가능하면 작은 grid 로 안전 선택. nested CV 는 일단 생략 (3 base 면 risk 낮음).

In [6]:
X_oof_arr  = P_oof.values
X_val_arr  = P_val.values
X_test_arr = P_test.values
y_oof_np   = y_oof_arr.values
y_val_np   = y_val_arr.values
y_test_np  = y_test_arr.values

# ElasticNetCV 로 alpha + l1_ratio 자동 선택 (5-fold)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('enet',   ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],   # 0=Ridge, 1=Lasso
        alphas=np.logspace(-6, 0, 30),
        cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
        random_state=SEED,
        n_jobs=-1,
        max_iter=20000,
        positive=False,    # 음수 weight 허용 (Ridge-like 효과 + 모델 간 보완)
    )),
])
pipe.fit(X_oof_arr, y_oof_np)

best_alpha    = pipe.named_steps['enet'].alpha_
best_l1_ratio = pipe.named_steps['enet'].l1_ratio_
stack_coef    = pipe.named_steps['enet'].coef_
stack_intercept = pipe.named_steps['enet'].intercept_

print(f'=== ElasticNet meta best ===')
print(f'  alpha    : {best_alpha:.6e}')
print(f'  l1_ratio : {best_l1_ratio:.4f}')
print(f'  coef (scaled space):')
for n, c in zip(BASE_PATHS, stack_coef):
    print(f'    {n:10s}: {c:+.4f}')
print(f'  intercept: {stack_intercept:+.6f}')

# predict + 음수 clip
stack_oof  = np.clip(pipe.predict(X_oof_arr),  0, None)
stack_val  = np.clip(pipe.predict(X_val_arr),  0, None)
stack_test = np.clip(pipe.predict(X_test_arr), 0, None)

rmse_stack_oof  = _rmse(stack_oof,  y_oof_np)
rmse_stack_val  = _rmse(stack_val,  y_val_np)
rmse_stack_test = _rmse(stack_test, y_test_np)

print(f'\n=== Stacking (ElasticNet + StandardScaler) RMSE ===')
print(f'  OOF :  {rmse_stack_oof:.6f}')
print(f'  val :  {rmse_stack_val:.6f}')
print(f'  test:  {rmse_stack_test:.6f}')
print(f'  음수 clip 비율: oof={(pipe.predict(X_oof_arr)<0).mean():.1%}, '
      f'val={(pipe.predict(X_val_arr)<0).mean():.1%}, test={(pipe.predict(X_test_arr)<0).mean():.1%}')

=== ElasticNet meta best ===
  alpha    : 2.807216e-05
  l1_ratio : 0.9000
  coef (scaled space):
    zit_bag   : +0.0008
    lgbm      : +0.0002
    enet      : -0.0000
  intercept: +0.002515

=== Stacking (ElasticNet + StandardScaler) RMSE ===
  OOF :  0.008249
  val :  0.005713
  test:  0.008413
  음수 clip 비율: oof=0.0%, val=0.0%, test=0.0%


## 7. 종합 비교 표

In [7]:
best_oof_row  = single_df.iloc[0]
best_val_row  = single_df.sort_values('val').iloc[0]
best_test_row = single_df.sort_values('test').iloc[0]

comparison = pd.DataFrame([
    {'method': f'best single (OOF) [{best_oof_row["model"]}]',
     'oof': best_oof_row['oof'], 'val': best_oof_row['val'], 'test': best_oof_row['test']},
    {'method': f'best single (val) [{best_val_row["model"]}]',
     'oof': best_val_row['oof'], 'val': best_val_row['val'], 'test': best_val_row['test']},
    {'method': f'best single (test)[{best_test_row["model"]}]',
     'oof': best_test_row['oof'], 'val': best_test_row['val'], 'test': best_test_row['test']},
    {'method': 'Blending SLSQP',
     'oof': rmse_blend_oof, 'val': rmse_blend_val, 'test': rmse_blend_test},
    {'method': 'Stacking ElasticNet',
     'oof': rmse_stack_oof, 'val': rmse_stack_val, 'test': rmse_stack_test},
])
print('=' * 90)
print(f'  Stacking 3-base ({zit_bag_name} + lgbm + enet)')
print('=' * 90)
print(comparison.to_string(index=False, float_format='%.6f'))
print('=' * 90)

# Δ vs 단일 best
for label, oof, val, test in [
    ('Blending  vs single OOF best ', rmse_blend_oof, rmse_blend_val, rmse_blend_test),
    ('Stacking  vs single OOF best ', rmse_stack_oof, rmse_stack_val, rmse_stack_test),
]:
    d_oof  = oof  - best_oof_row['oof']
    d_val  = val  - best_val_row['val']
    d_test = test - best_test_row['test']
    print(f'  {label}: Δoof={d_oof:+.6f}  Δval={d_val:+.6f}  Δtest={d_test:+.6f}')

  Stacking 3-base (bag_zit_combined_best + lgbm + enet)
                     method      oof      val     test
best single (OOF) [zit_bag] 0.008251 0.005710 0.008412
best single (val) [zit_bag] 0.008251 0.005710 0.008412
best single (test)[zit_bag] 0.008251 0.005710 0.008412
             Blending SLSQP 0.008256 0.005730 0.008425
        Stacking ElasticNet 0.008249 0.005713 0.008413
  Blending  vs single OOF best : Δoof=+0.000005  Δval=+0.000019  Δtest=+0.000014
  Stacking  vs single OOF best : Δoof=-0.000002  Δval=+0.000003  Δtest=+0.000002


## 8. 산출물 저장 (`_temp/stacking_3base/`)

In [8]:
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'stacking_3base')
os.makedirs(OUT_DIR, exist_ok=True)

def _save_unit(pred_arr, ids, y_arr, fname):
    out = pd.DataFrame({
        KEY_COL: ids,
        'pred':  pred_arr,
        'health': y_arr.reindex(ids).values if hasattr(y_arr, 'reindex') else y_arr,
    })
    out.to_csv(os.path.join(OUT_DIR, fname), index=False)

# stacking 산출물
_save_unit(stack_oof,  y_oof_arr.index.values,  y_oof_arr,  'oof_unit_stack.csv')
_save_unit(stack_val,  y_val_arr.index.values,  y_val_arr,  'val_unit_stack.csv')
_save_unit(stack_test, y_test_arr.index.values, y_test_arr, 'test_unit_stack.csv')

# blending baseline 산출물
_save_unit(blend_oof.set_index(KEY_COL).loc[y_oof_arr.index, 'pred'].values,
           y_oof_arr.index.values,  y_oof_arr,  'oof_unit_blend.csv')
_save_unit(blend_val.set_index(KEY_COL).loc[y_val_arr.index, 'pred'].values,
           y_val_arr.index.values,  y_val_arr,  'val_unit_blend.csv')
_save_unit(blend_test.set_index(KEY_COL).loc[y_test_arr.index, 'pred'].values,
           y_test_arr.index.values, y_test_arr, 'test_unit_blend.csv')

# 단일 RMSE + corr
single_df.to_csv(os.path.join(OUT_DIR, 'single_base_rmse.csv'), index=False)
corr_oof.to_csv(os.path.join(OUT_DIR, 'residual_corr_oof.csv'))
comparison.to_csv(os.path.join(OUT_DIR, 'comparison.csv'), index=False)

meta = {
    'base_models':    list(BASE_PATHS.keys()),
    'base_paths':     {k: BASE_PATHS[k] for k in BASE_PATHS},
    'zit_bag_chosen': zit_bag_name,
    'zit_bag_priority': ZIT_BAG_PRIORITY,
    'override_zit_bag': OVERRIDE_ZIT_BAG,
    'meta_learner':   {
        'type':       'ElasticNetCV(StandardScaler)',
        'alpha':      float(best_alpha),
        'l1_ratio':   float(best_l1_ratio),
        'coef':       {n: float(c) for n, c in zip(BASE_PATHS, stack_coef)},
        'intercept':  float(stack_intercept),
        'positive':   False,
        'max_iter':   20000,
    },
    'blending_slsqp': {
        'weights': {k: float(v) for k, v in w_blend.items()},
        'rmse_oof':  rmse_blend_oof,
        'rmse_val':  rmse_blend_val,
        'rmse_test': rmse_blend_test,
    },
    'stacking': {
        'rmse_oof':  rmse_stack_oof,
        'rmse_val':  rmse_stack_val,
        'rmse_test': rmse_stack_test,
    },
    'single_base': single_df.to_dict(orient='records'),
    'SEED':        int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\stacking_3base
  comparison.csv                              0.4 KB
  meta.json                                   1.8 KB
  oof_unit_blend.csv                        960.9 KB
  oof_unit_stack.csv                        970.0 KB
  residual_corr_oof.csv                       0.2 KB
  single_base_rmse.csv                        0.2 KB
  test_unit_blend.csv                       320.1 KB
  test_unit_stack.csv                       323.3 KB
  val_unit_blend.csv                        320.0 KB
  val_unit_stack.csv                        323.3 KB


## 9. 요약

In [9]:
print('=' * 90)
print(f' Stacking 3-base — {zit_bag_name} + lgbm + enet  |  Meta: ElasticNet + StandardScaler')
print('=' * 90)
print(comparison.to_string(index=False, float_format='%.6f'))
print('-' * 90)
print(f'  Meta best alpha    : {best_alpha:.4e}')
print(f'  Meta best l1_ratio : {best_l1_ratio:.4f}')
print(f'  Meta coef (scaled) : ' +
      ', '.join(f'{n}={c:+.3f}' for n, c in zip(BASE_PATHS, stack_coef)))
print(f'  Meta intercept     : {stack_intercept:+.6f}')
print('=' * 90)
print(f'  Stacking이 Blending보다 낮으면 비선형/음수 weight 효과 입증.')
print(f'  Stacking이 단일 best보다 낮으면 ensemble 효과 입증.')

 Stacking 3-base — bag_zit_combined_best + lgbm + enet  |  Meta: ElasticNet + StandardScaler
                     method      oof      val     test
best single (OOF) [zit_bag] 0.008251 0.005710 0.008412
best single (val) [zit_bag] 0.008251 0.005710 0.008412
best single (test)[zit_bag] 0.008251 0.005710 0.008412
             Blending SLSQP 0.008256 0.005730 0.008425
        Stacking ElasticNet 0.008249 0.005713 0.008413
------------------------------------------------------------------------------------------
  Meta best alpha    : 2.8072e-05
  Meta best l1_ratio : 0.9000
  Meta coef (scaled) : zit_bag=+0.001, lgbm=+0.000, enet=-0.000
  Meta intercept     : +0.002515
  Stacking이 Blending보다 낮으면 비선형/음수 weight 효과 입증.
  Stacking이 단일 best보다 낮으면 ensemble 효과 입증.
